# Outbound Auth (3LO) with Supabase as Inbound Auth

This tutorial is a variant of the standard **Outbound Auth 3LO** tutorial
(`05-Outbound_Auth_3lo`) that replaces **Amazon Cognito** with **Supabase Auth**
as the inbound identity provider (IDP).

The outbound side — Google Calendar access via OAuth2 3-Legged Auth — is
**identical** to the original tutorial. Only the inbound auth mechanism changes.

## What changes vs the Cognito version

| Aspect | Cognito version | This version (Supabase) |
|:-------|:----------------|:------------------------|
| Inbound IDP | Amazon Cognito User Pool | Supabase Auth |
| OIDC discovery URL | `cognito-idp.<region>.amazonaws.com/<pool-id>/.well-known/openid-configuration` | `<project>.supabase.co/auth/v1/.well-known/openid-configuration` |
| Authorizer field | `allowedClients` + Cognito App Client ID (matched to `client_id` claim) | `allowedAudience` + `"authenticated"` (matched to `aud` claim) |
| Get bearer token | `boto3` `initiate_auth` | `supabase-py` `sign_in_with_password` |
| JWT signing algorithm | RS256 (Cognito default) | **Must be changed to RS256 or ES256** (see prerequisites) |

## Tutorial Architecture

```
User
 │  (Supabase JWT — inbound auth)
 ▼
AgentCore Runtime  ──────►  customJWTAuthorizer
 │                             validates JWT via Supabase OIDC discovery
 ▼
Strands Agent (Claude Haiku 4.5)
 │
 ▼  get_calendar_events_today tool
AgentCore Identity  ──────►  @requires_access_token  (outbound auth 3LO)
 │                             google-cal-provider
 ▼
Google Calendar API
```

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Conversational |
| Agent type | Single |
| Agentic Framework | Strands Agents |
| LLM model | Anthropic Claude Haiku 4.5 |
| Inbound Auth | Supabase Auth (customJWTAuthorizer) |
| Outbound Auth | OAuth2 3LO — Google Calendar |
| SDK used | Amazon BedrockAgentCore Python SDK, boto3, supabase-py |

## Prerequisites

### AWS
- Python 3.10+
- AWS credentials configured
- Docker running

### Supabase (critical — do this before running any cells)

1. **Create a Supabase project** at https://supabase.com (free tier is sufficient).

2. **Change the JWT signing algorithm to RS256 or ES256.**
   The default algorithm is HS256 (symmetric). AgentCore's `customJWTAuthorizer`
   validates tokens by fetching public keys from the JWKS endpoint — this only
   works with an asymmetric algorithm.

   > **Supabase dashboard → Project Settings → API → JWT Settings →
   > JWT Algorithm → select RS256 or ES256 → Save**

3. **Create a test user** in Supabase Auth:
   > Authentication → Users → Add user → Create new user
   >
   > Use any email/password. You will store these in `.env` below.

4. **Copy your project credentials** from:
   > Project Settings → API
   > - **Project URL** (e.g. `https://xxxx.supabase.co`)
   > - **anon / public key** (safe to use client-side)

### Google Calendar (same as 05-Outbound_Auth_3lo)
- Register an OAuth2 app in Google Developer Console
- Enable the Google Calendar API
- Obtain a Client ID and Client Secret
- (You will register the AgentCore callback URL with Google in a later step)

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import sys
import os

current_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

In [ ]:
import subprocess
import dotenv
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

identity_client = boto_session.client("bedrock-agentcore-control")

## Step 1 — Configure Supabase as the Inbound IDP

Create a `.env` file in this directory with your Supabase project credentials
and the test user you created in the prerequisites.

> **Note:** `SUPABASE_ANON_KEY` is the public **anon** key, not the service role key.
> It is safe to include in a `.env` file used for local development.

In [ ]:
%%writefile .env
# Supabase inbound auth credentials
SUPABASE_URL="https://<your-project-ref>.supabase.co"
SUPABASE_ANON_KEY="<your-anon-key>"
SUPABASE_USER_EMAIL="<your-test-user-email>"
SUPABASE_USER_PASSWORD="<your-test-user-password>"

# Google OAuth2 outbound credentials (filled in Step 3)
GOOGLE_CLIENT_ID=""
GOOGLE_CLIENT_SECRET=""

Edit the `.env` file above with your real values, then run the cell below to
sign in and obtain an inbound bearer token.

In [ ]:
from supabase_auth_utils import setup_supabase_auth, reauthenticate_supabase_user

dotenv.load_dotenv(override=True)

supabase_config = setup_supabase_auth(
    supabase_url=os.environ["SUPABASE_URL"],
    supabase_anon_key=os.environ["SUPABASE_ANON_KEY"],
    email=os.environ["SUPABASE_USER_EMAIL"],
    password=os.environ["SUPABASE_USER_PASSWORD"],
)

# These two values wire into AgentCore's customJWTAuthorizer.
# Supabase access tokens have aud="authenticated", so we use that as the
# allowedAudience matches aud claim; use allowedClients only if token has client_id (OAuth server flow).
discovery_url = supabase_config["discovery_url"]
allowed_audience = "authenticated"   # matches the 'aud' claim in Supabase JWTs
bearer_token     = supabase_config["bearer_token"]

print(f"\ndiscovery_url : {discovery_url}")
print(f"allowed_audience: {allowed_audience}")
print("Supabase inbound auth configured ✓")

### Why `allowedAudience = "authenticated"`?

AgentCore's `customJWTAuthorizer` has two separate fields:
- `allowedClients` — validates the `client_id` JWT claim (used by Cognito)
- `allowedAudience` — validates the `aud` JWT claim (used here for Supabase)

Supabase JWTs issued via `signInWithPassword` always contain `aud: "authenticated"`
(the standard OIDC audience for Supabase). There is no `client_id` claim in this
flow, so we match on `aud`.

If you use the **Supabase OAuth Server** flow (Supabase acting as an OAuth AS
issuing tokens to a registered OAuth application), the token will also contain a
`client_id` claim equal to your OAuth app's client ID. In that case you would use
that client ID in `allowedClients` instead of `allowedAudience`.

## Step 2 — Configure Google for OAuth2 Outbound Auth (3LO)

Follow the same steps as in `05-Outbound_Auth_3lo` to register a Google OAuth2
app and enable the Google Calendar API:

1. Go to the [Google Developer Console](https://console.developers.google.com/)
2. Create a project and enable the **Google Calendar API**
3. Configure the **OAuth consent screen** (External audience, add your Gmail as test user)
4. Create **OAuth 2.0 Credentials** → Web application → copy the **Client ID** and **Client Secret**
5. Under **Data access**, add scope `https://www.googleapis.com/auth/calendar.readonly`

Then update the `.env` file with the Google credentials and reload it:

In [ ]:
# After editing .env with GOOGLE_CLIENT_ID and GOOGLE_CLIENT_SECRET, reload:
dotenv.load_dotenv(override=True)
print("GOOGLE_CLIENT_ID set:", bool(os.environ.get("GOOGLE_CLIENT_ID")))
print("GOOGLE_CLIENT_SECRET set:", bool(os.environ.get("GOOGLE_CLIENT_SECRET")))

## Step 3 — Create the Google OAuth2 Credential Provider

Register the Google OAuth2 credential provider with AgentCore Identity.
AgentCore returns a **callback URL** that you must register with Google.

In [ ]:
google_provider = identity_client.create_oauth2_credential_provider(
    **{
        "name": "google-cal-provider",
        "credentialProviderVendor": "GoogleOauth2",
        "oauth2ProviderConfigInput": {
            "googleOauth2ProviderConfig": {
                "clientId": os.environ["GOOGLE_CLIENT_ID"],
                "clientSecret": os.environ["GOOGLE_CLIENT_SECRET"],
            }
        },
    }
)

print(google_provider)
print(f"\ncallbackUrl: {google_provider['callbackUrl']}")
print("\n⚠️  Copy the callbackUrl above and add it to your Google OAuth2 app under")
print("   APIs & Services > Credentials > [your app] > Authorised redirect URIs")

## Step 4 — Register the AgentCore Callback URL in Google Console

1. Go to [Google Developer Console](https://console.developers.google.com/)
2. **APIs & Services → Credentials → [your OAuth 2.0 Client]**
3. Under **Authorised redirect URIs**, add the `callbackUrl` printed above
4. Click **Save**

This tells Google where to redirect the user's browser after they grant consent.

## Step 5 — Determine the Local OAuth2 Callback URL

The `oauth2_callback_server.py` runs locally (port 9090) to handle session
binding (see the `05-Outbound_Auth_3lo` tutorial for the full explanation).
The URL is passed into the agent container as the `CALLBACK_URL` env var.

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

oauth2_callback_url_for_agent = get_oauth2_callback_url()
print(f"Callback URL for agent: {oauth2_callback_url_for_agent}")

## Step 6 — Configure AgentCore Runtime Deployment

The key difference from the Cognito version is the `authorizer_configuration`:

- `discoveryUrl` → Supabase OIDC discovery URL
- `allowedAudience` → `["authenticated"]` (matches the `aud` claim in Supabase JWTs)

Everything else — entrypoint, ECR, execution role — is identical.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Region: {region}")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_google_3lo.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    memory_mode="NO_MEMORY",
    agent_name="strands_agent_supabase_3lo",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,          # Supabase OIDC discovery URL
            "allowedAudience": [allowed_audience],  # matches aud claim in Supabase JWTs
        }
    },
)
print(response)

## Step 7 — Review the AgentCore Configuration

In [ ]:
!cat .bedrock_agentcore.yaml

## Step 8 — Launch the Agent to AgentCore Runtime

This step:
1. Builds an ARM64 container via AWS CodeBuild and pushes to ECR
2. Creates the AgentCore Runtime endpoint
3. Retrieves the deployed agent's workload identity and registers the local
   OAuth2 callback server URL as an `allowedResourceOauth2ReturnUrl`
   (required for OAuth2 session binding)

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

launch_result = agentcore_runtime.launch(
    env_vars={"CALLBACK_URL": oauth2_callback_url_for_agent},
    auto_update_on_conflict=True,
)
print(launch_result)

if launch_result.agent_id:
    workload_name = launch_result.agent_id
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    allowed_resource_oauth_2_return_urls = (
        workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    )
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowedResourceOauth2ReturnUrls=[
            *allowed_resource_oauth_2_return_urls,
            oauth2_callback_url,
        ],
    )
    print(updated_workload_identity)

## Step 9 — Attach Extra IAM Policies to the Runtime Role

The auto-created execution role needs additional permissions:
- `bedrock-agentcore:GetResourceOauth2Token` — to trigger the 3LO consent flow
- `secretsmanager:GetSecretValue` — to read the Google OAuth2 client secret

In [ ]:
import json
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
runtime_response = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
runtime_role = runtime_response["roleArn"]
account = boto_session.client("sts").get_caller_identity().get("Account")

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockMarketplaceAccess",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe",
            ],
            "Resource": "*",
        },
        {
            "Sid": "Oauth2TokenAccess",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetResourceOauth2Token"],
            "Resource": "*",
        },
        {
            "Sid": "SecretsManagerAccess",
            "Effect": "Allow",
            "Action": ["secretsmanager:GetSecretValue"],
            "Resource": [
                f"arn:aws:secretsmanager:{region}:{account}:secret:"
                "bedrock-agentcore-identity!default/oauth2/google-cal-provider*"
            ],
        },
    ],
}

iam_client = boto3.client("iam", region_name=region)
iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)
print("Extra policies attached ✓")

## Step 10 — Wait for the Agent Endpoint to be READY

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)

print(f"Final status: {status}")

## Step 11 — Invoke the Agent

Before invoking:
1. The local `oauth2_callback_server.py` is started on port 9090
2. The Supabase bearer token is stored in the callback server (for session binding)
3. The agent is invoked with the Supabase JWT as the `bearer_token`

On the first invocation, the agent will trigger the 3LO consent flow and print
an **authorization URL**. Open that URL in your browser to grant Google Calendar
access. The callback server will complete the session binding automatically.
Re-invoke the agent to get the calendar response.

> **Note:** Supabase access tokens expire after ~1 hour. If you receive a 401,
> call `reauthenticate_supabase_user(supabase_config)` to get a fresh token.

In [ ]:
from oauth2_callback_server import (
    store_token_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
)

# Refresh the Supabase token in case it has expired
bearer_token = reauthenticate_supabase_user(supabase_config)

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    successfully_started = wait_for_oauth2_server_to_be_ready()
    if not successfully_started:
        print(
            "Failed to start OAuth2 callback server. "
            "See https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/"
            "oauth2-authorization-url-session-binding.html"
        )
    else:
        store_token_in_oauth2_callback_server(bearer_token)
        invoke_response = agentcore_runtime.invoke(
            {"prompt": "What is in my agenda for today? Highlight the main events!"},
            bearer_token=bearer_token,
        )
        print(invoke_response)
finally:
    oauth2_callback_server_process.terminate()

## Optional — Test with the Streamlit Chatbot

`chatbot_app_supabase.py` provides a chat UI that replaces the Cognito login
form with a Supabase email/password sign-in form. It reads the agent ARN from
`.bedrock_agentcore.yaml` and Supabase credentials from `.env`.

**Run from terminal:**
```bash
cd 01-tutorials/03-AgentCore-identity/14-supabase-outbound-auth-3lo/
python oauth2_callback_server.py -r <region> & streamlit run chatbot_app_supabase.py
```

Or run the cell below to launch from the notebook:

In [ ]:
from oauth2_callback_server import wait_for_oauth2_server_to_be_ready

notebook_dir = os.getcwd()

oauth2_callback_server_process = subprocess.Popen(
    [sys.executable, "oauth2_callback_server.py", "--region", region]
)

try:
    wait_for_oauth2_server_to_be_ready()

    process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "chatbot_app_supabase.py",
            "--server.port=8501",
            "--server.showEmailPrompt=false",
        ],
        cwd=notebook_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    for line in iter(process.stdout.readline, ""):
        if line:
            if "8501" in line:
                print("\nStreamlit app is ready!")
                print("\nURL: http://localhost:8501")
                print("Sign in with your Supabase test user credentials from .env")
                break

except KeyboardInterrupt:
    print("\nStopped.")
    oauth2_callback_server_process.terminate()
    process.terminate()
except Exception as e:
    print(f"Error starting Streamlit app: {e}")

## Cleanup (Optional)

Uncomment and run the cells below to delete the AgentCore Runtime and ECR repository.

In [ ]:
# launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# import boto3
# agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
# ecr_client = boto3.client('ecr', region_name=region)
#
# agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )
# ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1],
#     force=True,
# )

## Congratulations!

You have deployed a Strands agent on AgentCore Runtime that:
- Uses **Supabase Auth** as the inbound identity provider (JWT validation via OIDC)
- Accesses **Google Calendar** on behalf of the authenticated user via OAuth2 3LO

### Key differences from the Cognito version

| | Cognito | Supabase |
|---|---|---|
| Discovery URL | AWS-hosted per pool | `<project>.supabase.co/auth/v1/.well-known/openid-configuration` |
| Authorizer field | `allowedClients` (checks `client_id`) | `allowedAudience` (checks `aud`) |
| Get token | `boto3` `initiate_auth` | `supabase-py` `sign_in_with_password` |
| Token expiry | 2 hours | ~1 hour (configurable) |
| Algorithm setup | RS256 by default | **Must manually set RS256/ES256** |
| User management | Cognito console / CLI | Supabase dashboard / API |